# PLINK / GWAS QC Workflow Agent
## Colab 03 v3 — Stable Planning Loop

這版針對 v2 的真實執行結果修正：

- 主流程最後雖然 PASS，但用了 3 次 revision
- 每次 revision 會偶爾刪掉原本正確的步驟

### v3 改進
1. 由程式依 `study_info` 先建立 **Required Step Checklist**
2. Planner 在第一次生成時就必須覆蓋所有必要步驟
3. Revision 時：
   - 只修 Validator 指出的問題
   - 不得刪除未被指出有問題的步驟
   - 必須重新檢查 Required Step Checklist
4. 將 temperature 降為 `0.0`，減少不必要變動

## 1. 安裝套件

In [ ]:
!pip -q install groq sentence-transformers faiss-cpu

## 2. Imports + Groq Client

In [ ]:
import re
import json
import shutil
from pathlib import Path
from copy import deepcopy
from pprint import pprint

import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from groq import Groq
from google.colab import userdata

GROQ_API_KEY = userdata.get("Groq")
if not GROQ_API_KEY:
    raise ValueError("找不到 Colab Secret：Groq")

client = Groq(api_key=GROQ_API_KEY)

MODEL_NAME = "openai/gpt-oss-20b"
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

TOP_K_PER_QUERY = 3
MAX_PLAN_ITERATIONS = 3
SEED = 123

print("Groq client: READY")

## 3. Clone Wiki

In [ ]:
REPO_URL = "https://github.com/b156698-commits/PLINK-GWAS-Quality-Control-Wiki.git"
REPO_DIR = Path("/content/PLINK-GWAS-Quality-Control-Wiki")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

!git clone -q {REPO_URL} {REPO_DIR}

wiki_dir = REPO_DIR / "wiki"
wiki_files = sorted(wiki_dir.glob("*.md"))

print("Wiki files:", len(wiki_files))

## 4. Read + Chunk + FAISS

In [ ]:
def extract_source_ids(text):
    return sorted(set(re.findall(r"\bS\d{2}\b", text)))

def chunk_markdown(text, max_chars=1200):
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks, current = [], ""

    for p in paragraphs:
        candidate = p if not current else current + "\n\n" + p
        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                chunks.append(current)
            current = p

    if current:
        chunks.append(current)

    return chunks

documents = []
for file_path in wiki_files:
    text = file_path.read_text(encoding="utf-8")
    documents.append({
        "source_file": file_path.name,
        "source_ids": extract_source_ids(text),
        "text": text
    })

chunks = []
for doc in documents:
    for i, chunk_text in enumerate(chunk_markdown(doc["text"])):
        chunks.append({
            "chunk_id": f'{doc["source_file"]}::chunk_{i+1}',
            "source_file": doc["source_file"],
            "source_ids": doc["source_ids"],
            "text": chunk_text
        })

embedding_model = SentenceTransformer(EMBEDDING_MODEL)
texts = [c["text"] for c in chunks]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)
embeddings = np.asarray(embeddings, dtype="float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Documents:", len(documents))
print("Chunks:", len(chunks))
print("FAISS index:", index.ntotal)

## 5. Retrieval

In [ ]:
def retrieve(query, top_k=3):
    qv = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(qv, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        item = chunks[int(idx)].copy()
        item["score"] = float(score)
        results.append(item)
    return results

def deduplicate_results(results):
    best = {}
    for r in results:
        key = r["chunk_id"]
        if key not in best or r["score"] > best[key]["score"]:
            best[key] = r
    return sorted(best.values(), key=lambda x: x["score"], reverse=True)

def build_rag_context(results):
    blocks = []
    for i, r in enumerate(results, start=1):
        ids = ", ".join(r["source_ids"]) if r["source_ids"] else "N/A"
        blocks.append(
            f"[Retrieved Chunk {i}]\n"
            f"Source file: {r['source_file']}\n"
            f"Source IDs: {ids}\n\n"
            f"{r['text']}"
        )
    return "\n\n".join(blocks)

## 6. Study Information

In [ ]:
study_info = {
    "study_design": "case-control",
    "data_format": "plink_binary",
    "sample_size": 3000,
    "phenotype": "Disease Status",
    "covariates": ["Age", "Sex"],
    "need_pca": True,
    "need_relatedness": True,
    "association": "logistic"
}

pprint(study_info)

## 7. Required Step Checklist

這是 v3 的核心改動。

必要步驟不再完全交給 LLM 自己判斷，
而是先由 deterministic Python 規則依 `study_info` 建立。

In [ ]:
def build_required_steps(study_info):
    required = [
        "Load PLINK Binary Files",
        "Variant Missingness",
        "Sample Missingness",
        "MAF Filtering",
        "HWE Filtering",
        "LD Pruning"
    ]

    if study_info.get("need_relatedness"):
        required.append("Relatedness")

    if study_info.get("need_pca"):
        required.append("PCA")

    required.append("Association Analysis")
    return required

required_steps = build_required_steps(study_info)

print("Required steps:")
for i, step in enumerate(required_steps, start=1):
    print(f"{i}. {step}")

## 8. Multi-query Retrieval + Coverage

In [ ]:
topic_queries = [
    "PLINK binary files bed bim fam bfile",
    "PLINK variant missingness --geno and sample missingness --mind",
    "PLINK MAF filtering and HWE filtering",
    "PLINK LD pruning --indep-pairwise",
]

if study_info.get("need_relatedness"):
    topic_queries.append(
        "PLINK relatedness IBD PI_HAT --genome and LD pruning"
    )

if study_info.get("need_pca"):
    topic_queries.append(
        "PLINK PCA population structure eigenvec eigenval"
    )

topic_queries.append(
    "PLINK case-control logistic association phenotype covariate pheno-name covar-name"
)

all_results = []
for q in topic_queries:
    all_results.extend(
        retrieve(q, top_k=TOP_K_PER_QUERY)
    )

retrieved_docs = deduplicate_results(all_results)

def required_wiki_files(study_info):
    files = {
        "01-plink-basics.md",
        "02-missingness.md",
        "03-maf-hwe.md",
        "04-ld-pruning.md",
        "07-association.md",
    }

    if study_info.get("need_relatedness"):
        files.add("05-relatedness.md")

    if study_info.get("need_pca"):
        files.add("06-pca.md")

    return files

def force_add_required_chapters(study_info, results):
    required = required_wiki_files(study_info)
    present = {r["source_file"] for r in results}
    missing = required - present

    result_map = {r["chunk_id"]: r for r in results}

    for c in chunks:
        if c["source_file"] in missing:
            item = c.copy()
            item["score"] = 0.0
            result_map[item["chunk_id"]] = item

    fixed = list(result_map.values())
    fixed.sort(key=lambda x: x["score"], reverse=True)
    return fixed

retrieved_docs = force_add_required_chapters(
    study_info,
    retrieved_docs
)

required_files = required_wiki_files(study_info)
present_files = {r["source_file"] for r in retrieved_docs}
missing_after = required_files - present_files

print("RETRIEVAL COVERAGE:", "PASS" if not missing_after else "FAIL")
if missing_after:
    print(sorted(missing_after))

allowed_source_ids = sorted({
    sid
    for r in retrieved_docs
    for sid in r["source_ids"]
})

print("Allowed Source IDs:", allowed_source_ids)

rag_context = build_rag_context(retrieved_docs)

## 9. JSON Schema

In [ ]:
QC_PLAN_SCHEMA = {
    "type": "object",
    "properties": {
        "qc_plan": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "step": {"type": "integer"},
                    "name": {"type": "string"},
                    "purpose": {"type": "string"},
                    "plink_flag": {"type": "string"},
                    "example_setting": {
                        "type": ["string", "null"]
                    },
                    "note": {"type": "string"},
                    "source": {"type": "string"}
                },
                "required": [
                    "step",
                    "name",
                    "purpose",
                    "plink_flag",
                    "example_setting",
                    "note",
                    "source"
                ],
                "additionalProperties": False
            }
        }
    },
    "required": ["qc_plan"],
    "additionalProperties": False
}

## 10. Planner Prompt v3

In [ ]:
PLANNER_SYSTEM_PROMPT = '''
You are the planning component of a PLINK / GWAS QC Workflow Agent.

Generate a QC plan using only the retrieved Wiki context.

STRICT RULES:
- Include EVERY step listed in REQUIRED STEP CHECKLIST.
- Preserve the checklist order unless the Wiki context requires a more specific substep order.
- Every step MUST use one or more valid Source IDs supplied in ALLOWED SOURCE IDS.
- The source field MUST contain only IDs such as S02 or S05, S06.
- NEVER write "尚待驗證", "unknown", "N/A", or prose in the source field.
- Do not invent PLINK flags.
- Threshold values are examples, not universal standards.
- LD pruning must precede relatedness/PCA when requested.
- Do not claim the final genome-wide association scan must use only LD-pruned SNPs.
- PCA must precede association when requested.
- Return only the requested structured output.
'''.strip()

## 11. Generate Initial Plan

In [ ]:
def generate_qc_plan(
    study_info,
    required_steps,
    rag_context,
    allowed_source_ids
):
    prompt = (
        "STUDY INFORMATION\n"
        + json.dumps(study_info, ensure_ascii=False, indent=2)
        + "\n\nREQUIRED STEP CHECKLIST\n"
        + json.dumps(required_steps, ensure_ascii=False, indent=2)
        + "\n\nALLOWED SOURCE IDS\n"
        + json.dumps(allowed_source_ids, ensure_ascii=False)
        + "\n\nRETRIEVED WIKI CONTEXT\n"
        + rag_context
        + "\n\nGenerate the complete QC plan."
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        seed=SEED,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "gwas_qc_plan",
                "strict": True,
                "schema": QC_PLAN_SCHEMA
            }
        }
    )

    data = json.loads(response.choices[0].message.content)
    return data["qc_plan"]

qc_plan = generate_qc_plan(
    study_info,
    required_steps,
    rag_context,
    allowed_source_ids
)

pprint(qc_plan)

## 12. Validator v3

In [ ]:
def normalize_name(name):
    return str(name).strip().lower()

def find_step_index(plan, keyword):
    keyword = keyword.lower()
    for i, step in enumerate(plan):
        if keyword in normalize_name(step.get("name", "")):
            return i
    return None

def parse_source_ids(source_text):
    return [
        x.strip()
        for x in str(source_text).split(",")
        if x.strip()
    ]

def has_required_step(plan_names, required_name):
    req = required_name.lower()

    aliases = {
        "load plink binary files": ["load plink", "plink binary", "input"],
        "variant missingness": ["variant missingness"],
        "sample missingness": ["sample missingness"],
        "maf filtering": ["maf"],
        "hwe filtering": ["hwe"],
        "ld pruning": ["ld pruning"],
        "relatedness": ["relatedness"],
        "pca": ["pca"],
        "association analysis": ["association"]
    }

    patterns = aliases.get(req, [req])
    return any(
        any(p in n for p in patterns)
        for n in plan_names
    )

def validate_qc_plan(
    study_info,
    required_steps,
    plan,
    allowed_source_ids
):
    errors = []

    if not isinstance(plan, list) or len(plan) == 0:
        return ["QC plan is empty."]

    names = [
        normalize_name(step.get("name", ""))
        for step in plan
    ]

    # A. Required checklist coverage
    for required in required_steps:
        if not has_required_step(names, required):
            errors.append(
                f"Missing required step: {required}."
            )

    # B. Order / logic
    ld_idx = find_step_index(plan, "ld pruning")
    rel_idx = find_step_index(plan, "relatedness")
    pca_idx = find_step_index(plan, "pca")
    assoc_idx = find_step_index(plan, "association")

    if study_info.get("need_relatedness") and rel_idx is not None:
        if ld_idx is None:
            errors.append("Relatedness is requested, but LD Pruning is missing.")
        elif ld_idx > rel_idx:
            errors.append("LD Pruning should occur before Relatedness.")

    if study_info.get("need_pca") and pca_idx is not None:
        if ld_idx is None:
            errors.append("PCA is requested, but LD Pruning is missing.")
        elif ld_idx > pca_idx:
            errors.append("LD Pruning should occur before PCA.")

        if assoc_idx is not None and pca_idx > assoc_idx:
            errors.append("PCA should occur before Association Analysis.")

    # C. Source
    allowed = set(allowed_source_ids)

    for i, step in enumerate(plan, start=1):
        raw_source = step.get("source", "")
        source_ids = parse_source_ids(raw_source)

        if not source_ids:
            errors.append(
                f"Step {i} ({step.get('name', 'Unknown')}) has no source."
            )
            continue

        for sid in source_ids:
            if not re.fullmatch(r"S\d{2}", sid):
                errors.append(
                    f"Step {i} ({step.get('name', 'Unknown')}) "
                    f"has invalid source format: {sid}"
                )
            elif sid not in allowed:
                errors.append(
                    f"Step {i} ({step.get('name', 'Unknown')}) "
                    f"uses source not present in retrieved context: {sid}"
                )

    # D. Threshold wording
    bad_phrases = [
        "standard threshold",
        "universal threshold",
        "must use",
        "fixed threshold"
    ]

    for i, step in enumerate(plan, start=1):
        combined = " ".join([
            str(step.get("example_setting", "")),
            str(step.get("note", "")),
            str(step.get("purpose", ""))
        ]).lower()

        for phrase in bad_phrases:
            if phrase in combined:
                errors.append(
                    f"Step {i} ({step.get('name', 'Unknown')}) "
                    f"may describe an example threshold as universal: {phrase}"
                )

    # E. Final GWAS pruning misuse
    for i, step in enumerate(plan, start=1):
        if "association" in normalize_name(step.get("name", "")):
            combined = " ".join([
                str(step.get("purpose", "")),
                str(step.get("note", ""))
            ]).lower()

            risky_patterns = [
                "only ld-pruned",
                "only use ld-pruned",
                "final gwas uses pruned",
                "final gwas only",
                "only the ld-pruned"
            ]

            if any(p in combined for p in risky_patterns):
                errors.append(
                    f"Step {i} (Association Analysis) may incorrectly "
                    "restrict the final GWAS to the LD-pruned SNP subset."
                )

    return errors

## 13. Initial Validation

In [ ]:
plan_errors = validate_qc_plan(
    study_info,
    required_steps,
    qc_plan,
    allowed_source_ids
)

print("PLAN VALIDATION:", "PASS" if not plan_errors else "FAIL")
for i, e in enumerate(plan_errors, start=1):
    print(f"{i}. {e}")

## 14. Stable Revision Function

In [ ]:
def revise_qc_plan(
    study_info,
    required_steps,
    rag_context,
    allowed_source_ids,
    current_plan,
    errors
):
    prompt = (
        "STUDY INFORMATION\n"
        + json.dumps(study_info, ensure_ascii=False, indent=2)
        + "\n\nREQUIRED STEP CHECKLIST\n"
        + json.dumps(required_steps, ensure_ascii=False, indent=2)
        + "\n\nALLOWED SOURCE IDS\n"
        + json.dumps(allowed_source_ids, ensure_ascii=False)
        + "\n\nCURRENT PLAN\n"
        + json.dumps(current_plan, ensure_ascii=False, indent=2)
        + "\n\nVALIDATOR ERRORS\n"
        + json.dumps(errors, ensure_ascii=False, indent=2)
        + "\n\nPATCH THE PLAN MINIMALLY. "
          "Fix every validator error, but DO NOT remove or rewrite unrelated valid steps. "
          "The final plan MUST still contain EVERY item in REQUIRED STEP CHECKLIST. "
          "For source fields, use ONLY valid Source IDs from ALLOWED SOURCE IDS. "
          "Return the full corrected plan."
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        seed=SEED,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "revised_gwas_qc_plan",
                "strict": True,
                "schema": QC_PLAN_SCHEMA
            }
        }
    )

    data = json.loads(response.choices[0].message.content)
    return data["qc_plan"]

## 15. Stable Planning Loop

In [ ]:
state = {
    "study_info": study_info,
    "required_steps": required_steps,
    "retrieved_docs": retrieved_docs,
    "allowed_source_ids": allowed_source_ids,
    "qc_plan": qc_plan,
    "plan_errors": [],
    "plan_iteration": 0
}

while True:
    state["plan_errors"] = validate_qc_plan(
        state["study_info"],
        state["required_steps"],
        state["qc_plan"],
        state["allowed_source_ids"]
    )

    print("=" * 70)
    print(f'Validation round {state["plan_iteration"] + 1}')

    if not state["plan_errors"]:
        print("PLAN VALIDATION: PASS")
        break

    print("PLAN VALIDATION: FAIL")
    for i, e in enumerate(state["plan_errors"], start=1):
        print(f"{i}. {e}")

    if state["plan_iteration"] >= MAX_PLAN_ITERATIONS:
        print("\nSTOP / NEED HUMAN REVIEW")
        break

    state["qc_plan"] = revise_qc_plan(
        state["study_info"],
        state["required_steps"],
        rag_context,
        state["allowed_source_ids"],
        state["qc_plan"],
        state["plan_errors"]
    )

    state["plan_iteration"] += 1

print("\nFinal revision count:", state["plan_iteration"])

## 16. Demo — FAIL → PASS

故意：
- 刪除 Relatedness
- 把第一個 source 改成 `尚待驗證`

確認 stable revision 不會破壞其他已正確步驟。

In [ ]:
demo_bad_plan = deepcopy(state["qc_plan"])

demo_bad_plan = [
    x for x in demo_bad_plan
    if "relatedness" not in normalize_name(x.get("name", ""))
]

if demo_bad_plan:
    demo_bad_plan[0]["source"] = "尚待驗證"

demo_errors = validate_qc_plan(
    study_info,
    required_steps,
    demo_bad_plan,
    allowed_source_ids
)

print("DEMO VALIDATION:", "FAIL" if demo_errors else "PASS")
for i, e in enumerate(demo_errors, start=1):
    print(f"{i}. {e}")

demo_revised_plan = revise_qc_plan(
    study_info,
    required_steps,
    rag_context,
    allowed_source_ids,
    demo_bad_plan,
    demo_errors
)

demo_recheck = validate_qc_plan(
    study_info,
    required_steps,
    demo_revised_plan,
    allowed_source_ids
)

print("\nAFTER REVISION:", "PASS" if not demo_recheck else "FAIL")
for e in demo_recheck:
    print("-", e)

# 17. v3 完成標準

理想結果：

```text
RETRIEVAL COVERAGE: PASS

Validation round 1
PLAN VALIDATION: PASS
```

或最多只需要 1 次 revision：

```text
Validation round 1
FAIL

Validation round 2
PASS
```

最後 Demo：

```text
DEMO VALIDATION: FAIL
AFTER REVISION: PASS
```

### 為什麼 v3 比 v2 更合理？

v2 是：
> LLM 自己決定整份 plan 長什麼樣 → Validator 抓錯 → LLM 整份重寫

v3 是：
> Python 先決定 study-specific required steps → LLM 填內容 → Validator 抓錯 → LLM 最小幅度修正

因此 deterministic workflow 負責「不能缺什麼」，
LLM 負責「如何根據知識庫填寫內容」。

如果 v3 穩定後，再進入：
**04 — Command Generator + Command Validator**